<table align="left">
  <td><a target="_blank" href="https://colab.research.google.com/github/marcoteran/ml/blob/master/notebooks/01_ml_flujo_basico_fraude.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></td>
  <td><a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marcoteran/ml/blob/master/notebooks/01_ml_flujo_basico_fraude.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle"/></a></td>
</table>
<br><br>

# Flujo básico de Machine Learning: de los datos crudos a la inferencia

**Curso:** Aprendizaje Automático — SI7009 - 1 (5553)
**Sesión 1 · Parte 1:** Evaluación moderna, validación e hiperparámetros
**Universidad:** EAFIT
**Profesor:** Andrés Vásquez Restrepo
**Dataset:** Credit Card Transactions Fraud Detection (Kaggle `kartik2112/fraud-detection`)

---

**Objetivo:** recorrer el flujo completo sin depender de un modelo en particular: separar los datos, explorarlos, medir bien con clases raras, construir un **pipeline** que transforme siempre igual, validar, elegir el threshold y **usar el modelo sobre datos que nunca vio**.

El modelo es deliberadamente simple (regresión logística). La parte 2 reutiliza la misma preparación para comparar árboles, ensembles y boosting.

1. [Cómo ejecutar](#1) · 2. [Caso](#2) · 3. [Configuración](#3) · 4. [Datos](#4) · 5. [EDA](#5) · 6. [Outliers](#6) · 7. [Features](#7) · 8. [Métricas y baselines](#8) · 9. [Pipeline](#9) · 10. [Validación cruzada](#10) · 11. [Threshold](#11) · 12. [Hiperparámetros](#12) · 13. [Modelo final](#13) · 14. [Inferencia](#14) · 15. [Ejercicios y parte 2](#15)

<a name="1"></a>
## 1. Cómo ejecutar

**Local con `uv` (recomendado)**, desde la raíz del repositorio:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh   # instalar uv (una sola vez)
uv sync                                          # entorno con las versiones de uv.lock
uv run jupyter lab notebooks/01_ml_flujo_basico_fraude.ipynb
```

En VS Code basta elegir el kernel `.venv`.

**Google Colab:** botón *Abrir en Colab* y ejecutar la celda siguiente, que instala lo que falta. Los datos se descargan con `kagglehub` (dataset público); si Kaggle pide autenticación, suba `fraudTrain.csv` y `fraudTest.csv` a `/content/data/`.

> **Colab borra `/content` al cerrar la sesión.** Con `USE_GOOGLE_DRIVE = True` (celda 3.1) los artefactos se guardan en `MyDrive/SI7009_ML/artifacts`; si no, la sección 13 los descarga. El notebook 02 no depende de ellos: si no los encuentra, reconstruye el baseline.

In [ ]:
# =============================================================================
# 1.1 Environment detection and cloud install
# =============================================================================
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn>=1.5",
                    "kagglehub[pandas-datasets]>=1.0.2", "joblib"], check=True)
    print("Cloud environment: dependencies installed.")
else:
    print("Local environment: dependencies come from pyproject.toml / uv.lock (run `uv sync`).")

<a name="2"></a>
## 2. El caso

Un banco quiere **priorizar la revisión** de transacciones con tarjeta. Cada transacción trae datos crudos: fecha y hora, monto, categoría, ubicación del cliente y del comercio, fecha de nacimiento.

- **Clase positiva:** `is_fraud = 1`, menos del 1 % de los casos.
- **Costos (slides):** dejar pasar un fraude $C_{FN} = 500$; una falsa alarma $C_{FP} = 10$.

| Regla | Por qué |
|---|---|
| El test se separa **antes** del EDA y se usa **una vez** | mirarlo sesga las decisiones |
| Todo lo que aprende de los datos va **dentro del pipeline** | se ajusta solo con train y se repite igual en producción |
| Métrica principal **PR-AUC**; accuracy solo como trampa | con 0,6 % de positivos, accuracy no discrimina |
| El threshold se elige en validación | es un hiperparámetro de decisión |
| `fraudTest.csv` es **producción** | se usa solo en la sección 14 |

<a name="3"></a>
## 3. Configuración

In [ ]:
# =============================================================================
# 3.1 Imports, configuration and helpers
# =============================================================================
from __future__ import annotations

import json
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import loguniform
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, make_scorer, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV, StratifiedKFold,
                                     TunedThresholdClassifierCV, cross_val_predict, cross_val_score,
                                     train_test_split)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FAST_DEMO_MODE = True          # stratified sample of the history: runs in about a minute
FAST_SAMPLE_SIZE = 150_000
TARGET = "is_fraud"
C_FN, C_FP = 500, 10
N_SPLITS = 5

USE_GOOGLE_DRIVE = True        # only used in Colab: /content is wiped between sessions
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/SI7009_ML/artifacts")
elif IN_KAGGLE:
    ARTIFACT_DIR = Path("/kaggle/working/artifacts")
else:
    ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def expected_cost(y_true, y_pred) -> float:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return C_FN * fn + C_FP * fp


def binary_report(y_true, proba, threshold: float = 0.5, name: str = "") -> dict:
    y_pred = (np.asarray(proba) >= threshold).astype(int)
    return {"model": name, "threshold": threshold,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "pr_auc": average_precision_score(y_true, proba),
            "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(proba)) > 1 else 0.5,
            "alerts": int(y_pred.sum()), "expected_cost": expected_cost(y_true, y_pred)}


print(f"pandas {pd.__version__} | scikit-learn {sklearn.__version__} | artifacts: {ARTIFACT_DIR.resolve()}")

<a name="4"></a>
## 4. Datos: cargar y separar antes de mirar

- `fraudTrain.csv`: **historia** (2019 – mitad de 2020). Se usa para entrenar, validar y probar.
- `fraudTest.csv`: transacciones **posteriores**. Hace de producción.

**Columnas que no entran al modelo:** identificadores (`trans_num`, índice), datos personales (`cc_num`, `first`, `last`, `street`), duplicados (`unix_time`) y variables de cientos de valores (`merchant`, `city`, `job`, `zip`). Regla: **el modelo solo usa lo que llegará en producción al momento de decidir.**

El split se hace **antes** del EDA, estratificado para conservar la proporción de fraudes.

In [ ]:
# =============================================================================
# 4.1 Locate (or download) the data
# =============================================================================
DATASET_SLUG = "kartik2112/fraud-detection"
FILES = {"history": "fraudTrain.csv", "production": "fraudTest.csv"}
CANDIDATE_DIRS = [Path("data"), Path("../data"), Path("/content/data"), Path("/kaggle/input/fraud-detection")]


def find_or_download_fraud_data() -> Path:
    for folder in CANDIDATE_DIRS:
        if all((folder / name).exists() for name in FILES.values()):
            return folder
    import kagglehub  # public dataset
    return Path(kagglehub.dataset_download(DATASET_SLUG))


DATA_DIR = find_or_download_fraud_data()
print("Data folder:", DATA_DIR)

In [ ]:
# =============================================================================
# 4.2 SHARED PREP BLOCK (1/3, copiar igual en 02): load, clean, split
# =============================================================================
# Requires RANDOM_STATE, TARGET, FAST_DEMO_MODE, FAST_SAMPLE_SIZE (cell 3.1) and DATA_DIR, FILES (cell 4.1).
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_COLUMNS = [                      # the raw schema production must send
    "trans_date_trans_time", "category", "amt", "gender", "state",
    "lat", "long", "city_pop", "dob", "merch_lat", "merch_long",
]


def load_fraud_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, usecols=INPUT_COLUMNS + [TARGET])


def load_history(fast: bool = FAST_DEMO_MODE) -> pd.DataFrame:
    df = load_fraud_csv(DATA_DIR / FILES["history"])
    if fast and len(df) > FAST_SAMPLE_SIZE:
        df, _ = train_test_split(df, train_size=FAST_SAMPLE_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE)
    return df.reset_index(drop=True)


history = load_history()
X_train, X_test, y_train, y_test = train_test_split(
    history[INPUT_COLUMNS], history[TARGET], test_size=0.20, stratify=history[TARGET], random_state=RANDOM_STATE
)
print(f"history: {history.shape} | fraud rate {history[TARGET].mean():.3%}")
print(f"train: {X_train.shape} ({y_train.sum()} frauds) | test: {X_test.shape} ({y_test.sum()} frauds)")
X_train.head(3)

<a name="5"></a>
## 5. EDA (solo con train)

In [ ]:
# =============================================================================
# 5.1 Structure and the two strongest signals: amount and hour
# =============================================================================
eda = X_train.assign(**{TARGET: y_train})
ts = pd.to_datetime(eda["trans_date_trans_time"])
eda = eda.assign(hour=ts.dt.hour, age=(ts - pd.to_datetime(eda["dob"])).dt.days / 365.25,
                 log_amt=np.log1p(eda["amt"]))
print(f"nulls: {eda.isna().sum().sum()} | duplicated rows: {eda.duplicated().sum()} | fraud rate: {y_train.mean():.3%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
bins = np.linspace(0, eda["log_amt"].max(), 60)
for label, name in [(0, "legit"), (1, "fraud")]:
    axes[0].hist(eda.loc[eda[TARGET] == label, "log_amt"], bins=bins, density=True, alpha=0.6, label=name)
axes[0].set(title="Monto (log1p) por clase", xlabel="log(1 + amt)", ylabel="densidad")
axes[0].legend()
(eda.groupby("hour")[TARGET].mean() * 100).plot(ax=axes[1], marker="o")
axes[1].set(title="% fraude por hora", xlabel="hora", ylabel="% fraude")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 5.2 Fraud rate by category and age
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
(eda.groupby("category")[TARGET].mean() * 100).sort_values().plot.barh(ax=axes[0])
axes[0].set(title="% fraude por categoría", xlabel="% fraude", ylabel="")
(eda.groupby(pd.cut(eda["age"], [0, 25, 35, 50, 65, 100]), observed=True)[TARGET].mean() * 100).plot.bar(ax=axes[1], rot=0)
axes[1].set(title="% fraude por edad", xlabel="edad", ylabel="% fraude")
plt.tight_layout()
plt.show()

**Lectura técnica**

- **Monto:** los fraudes se concentran en montos medios y altos; en escala log las clases se separan.
- **Hora:** el fraude se dispara de noche (22–3 h) y es casi nulo el resto del día. La relación **no es lineal**, así que la hora se tratará como **categórica**.
- **Categoría:** las compras en línea (`shopping_net`, `misc_net`) tienen varias veces la tasa promedio. La edad aporta poco.

<a name="6"></a>
## 6. Outliers: ¿ruido o señal?

Reglas de las slides: **IQR** ($x > Q_3 + 1{,}5\,IQR$) y **3σ** ($x > \mu + 3\sigma$). En fraude la pregunta clave es: **¿qué pasa con el target dentro de los outliers?**

In [ ]:
# =============================================================================
# 6.1 Amount outliers and the fraud rate inside them
# =============================================================================
amt = eda["amt"]
q1, q3 = amt.quantile([0.25, 0.75])
rows = []
for rule, upper in [("IQR", q3 + 1.5 * (q3 - q1)), ("3 sigma", amt.mean() + 3 * amt.std())]:
    is_out = amt > upper
    rows.append({"rule": rule, "upper_limit": upper, "share_of_rows": is_out.mean(),
                 "fraud_rate_inside": eda.loc[is_out, TARGET].mean(),
                 "share_of_all_frauds": is_out[eda[TARGET] == 1].mean()})
print(f"base fraud rate: {y_train.mean():.3%} | max fraud amount: {amt[eda[TARGET] == 1].max():,.0f} | max amount: {amt.max():,.0f}")
pd.DataFrame(rows)

**Lectura técnica: en fraude, el outlier suele ser la señal.**

- La mayoría de los fraudes son outliers por IQR: **borrar outliers borraría casi todo el fraude**.
- Pero outlier ≠ fraude: los montos más extremos (miles) son legítimos.
- En producción no se puede "eliminar" una transacción: hay que puntuarla.

Decisión: **no se borran filas**. Se usa `log(1 + amt)` y se **recorta** (clip) el monto a un percentil **aprendido en train**, como un paso del pipeline.

<a name="7"></a>
## 7. Features reutilizables en producción

Todo lo que se calcula sobre los datos crudos (hora, edad, distancia, log del monto, clip) debe correr **igual** al entrenar y al predecir. Por eso se escribe como **transformadores de scikit-learn**:

- `FraudFeatureBuilder` no aprende nada en `fit`: calcula columnas fila a fila.
- `QuantileClipper` sí aprende: el límite del clip sale de train y queda guardado en el objeto.

`joblib` guarda **una referencia** a estas clases, no su código. Quien cargue el modelo (un servicio, el notebook 02) debe tener esta celda definida antes de `joblib.load`; en un sistema real iría empaquetada como librería.

In [ ]:
# =============================================================================
# 7.1 SHARED PREP BLOCK (2/3, copiar igual en 02): raw schema and reusable transformers
# =============================================================================
# Self-contained: this cell must be defined before joblib.load() of the artifact.
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

INPUT_SCHEMA = {
    "trans_date_trans_time": "datetime string",
    "category": "string",
    "amt": "float",
    "gender": "string",
    "state": "string",
    "lat": "float",
    "long": "float",
    "city_pop": "int",
    "dob": "date string",
    "merch_lat": "float",
    "merch_long": "float",
}
NUMERIC_FEATURES = ["amt", "log_amt", "log_city_pop", "age", "distance_km"]
CATEGORICAL_FEATURES = ["category", "gender", "state", "hour", "day_of_week"]


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


class FraudFeatureBuilder(BaseEstimator, TransformerMixin):
    """Raw transaction columns -> model features. Stateless: learns nothing in fit."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        ts = pd.to_datetime(X["trans_date_trans_time"], errors="coerce")
        dob = pd.to_datetime(X["dob"], errors="coerce")
        amt = pd.to_numeric(X["amt"], errors="coerce")
        return pd.DataFrame({
            "amt": amt,
            "log_amt": np.log1p(amt.clip(lower=0)),
            "log_city_pop": np.log1p(pd.to_numeric(X["city_pop"], errors="coerce")),
            "age": (ts - dob).dt.days / 365.25,
            "distance_km": haversine_km(X["lat"], X["long"], X["merch_lat"], X["merch_long"]),
            "category": X["category"].astype("object"),
            "gender": X["gender"].astype("object"),
            "state": X["state"].astype("object"),
            "hour": ts.dt.hour.astype("float").astype("Int64").astype("object"),
            "day_of_week": ts.dt.dayofweek.astype("float").astype("Int64").astype("object"),
        }, index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.array(NUMERIC_FEATURES + CATEGORICAL_FEATURES, dtype=object)


class QuantileClipper(BaseEstimator, TransformerMixin):
    """Clip each column to [lower, upper] quantiles learned on the training data."""

    def __init__(self, lower: float | None = None, upper: float | None = 0.999):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_ = X.quantile(self.lower) if self.lower is not None else None
        self.upper_ = X.quantile(self.upper) if self.upper is not None else None
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_)
        return X.clip(lower=self.lower_, upper=self.upper_, axis=1)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_


def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Check the raw input before scoring: required columns present, extra columns dropped."""
    missing = [c for c in INPUT_SCHEMA if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")
    extra = [c for c in df.columns if c not in INPUT_SCHEMA]
    if extra:
        print(f"[validate_schema] ignoring extra columns: {extra}")
    return df[list(INPUT_SCHEMA)]

In [ ]:
# =============================================================================
# 7.2 Raw rows in, model features out
# =============================================================================
FraudFeatureBuilder().fit_transform(X_train.head(3))

<a name="8"></a>
## 8. Métricas y baselines

| Métrica | Pregunta | Fórmula |
|---|---|---|
| Precision | de las alertas, ¿cuántas eran fraude? | $TP/(TP+FP)$ |
| Recall | de los fraudes, ¿cuántos detectamos? | $TP/(TP+FN)$ |
| PR-AUC | calidad del ranking sobre la clase positiva (azar = prevalencia) | área bajo precision–recall |
| ROC-AUC | separación global (azar = 0,5) | área bajo ROC |
| Costo | ¿cuánto cuesta la decisión? | $C_{FN}\,FN + C_{FP}\,FP$ |

Un baseline es la **referencia** que cualquier modelo debe superar. Aquí se miran en test solo para ilustrar; desde la sección 10 todo se decide con CV en train.

In [ ]:
# =============================================================================
# 8.1 Trivial, random and business-rule baselines
# =============================================================================
always_legit = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
stratified = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE).fit(X_train, y_train)
amt_p99 = X_train["amt"].quantile(0.99)               # learned on train only

baselines = pd.DataFrame([
    binary_report(y_test, always_legit.predict_proba(X_test)[:, 1], 0.5, "dummy: always legit"),
    binary_report(y_test, stratified.predict_proba(X_test)[:, 1], 0.5, "dummy: stratified"),
    binary_report(y_test, (X_test["amt"] >= amt_p99).astype(float).to_numpy(), 0.5, "rule: amt >= p99"),
])
baselines

**Lectura técnica:** el "siempre legítimo" tiene la **mejor accuracy** (≈ 99,4 %) y detecta **cero** fraudes. La regla de monto ya captura una parte con pocas alertas: un modelo que no la supere no se justifica.

<a name="9"></a>
## 9. Pipeline de procesamiento

```
datos crudos ─► FraudFeatureBuilder ─► numéricas: imputar ─► clip ─► escalar ─┐
                                   └─► categóricas: imputar ─► one-hot ─────┴─► modelo
```

Cada paso que **aprende** (mediana, límite del clip, media y desviación, lista de categorías) lo hace en `fit` **solo con train**; en `predict` solo aplica lo aprendido. Eso garantiza que los datos nuevos lleguen al modelo con **la misma escala y las mismas columnas** que en entrenamiento.

**MAL:** `StandardScaler().fit(X)` con todos los datos antes de separar o de la CV (leakage). **BIEN:** el preprocesamiento dentro del pipeline.

In [ ]:
# =============================================================================
# 9.1 SHARED PREP BLOCK (3/3, copiar igual en 02): preprocessing + pipeline builders
# =============================================================================
# In 02 only the `model` argument changes: same features, same preprocessing.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor(clip_upper: float | None = 0.999, min_frequency: int = 20) -> Pipeline:
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clip", QuantileClipper(upper=clip_upper)),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_frequency, sparse_output=False)),
    ])
    columns = ColumnTransformer([("num", numeric, NUMERIC_FEATURES), ("cat", categorical, CATEGORICAL_FEATURES)],
                                verbose_feature_names_out=False)
    return Pipeline([("features", FraudFeatureBuilder()), ("columns", columns)])


def build_pipeline(model=None, **prep_kwargs) -> Pipeline:
    model = model if model is not None else LogisticRegression(max_iter=2000)
    return Pipeline([("prep", build_preprocessor(**prep_kwargs)), ("model", model)]).set_output(transform="pandas")


build_pipeline()

In [ ]:
# =============================================================================
# 9.2 Same columns, train scale — and the serious baseline
# =============================================================================
pipe = build_pipeline().fit(X_train, y_train)
Z_train = pipe.named_steps["prep"].transform(X_train)
Z_test = pipe.named_steps["prep"].transform(X_test)
print(f"raw columns: {X_train.shape[1]} -> model features: {Z_train.shape[1]} (train) / {Z_test.shape[1]} (test)")
print("same columns, same order:", list(Z_train.columns) == list(Z_test.columns))
print("scaled amt mean/std  train:", Z_train["amt"].agg(["mean", "std"]).round(3).tolist(),
      "| test:", Z_test["amt"].agg(["mean", "std"]).round(3).tolist(), "(test uses the train scale)")

baselines = pd.concat([baselines, pd.DataFrame([binary_report(y_test, pipe.predict_proba(X_test)[:, 1], 0.5, "logreg pipeline")])],
                      ignore_index=True)
baselines

**Lectura técnica:** la regresión logística mejora mucho la PR-AUC, pero con threshold 0,5 casi no alerta y **su costo es peor que el de la regla**. El 0,5 no sirve con clases raras (sección 11).

<a name="10"></a>
## 10. Validación cruzada (hands-on 1)

Un solo split da un número; la CV da **media y dispersión**. Con clases raras, `KFold` reparte los positivos de forma desigual; `StratifiedKFold` conserva la prevalencia en cada fold. El pipeline completo se reajusta en cada fold, así que no hay leakage.

In [ ]:
# =============================================================================
# 10.1 KFold vs StratifiedKFold: positives per fold and PR-AUC spread
# =============================================================================
rows = []
for name, cv in {"KFold": KFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
                 "StratifiedKFold": StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE)}.items():
    scores = cross_val_score(build_pipeline(), X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1)
    rows.append({"cv": name, "positives_per_fold": [int(y_train.iloc[v].sum()) for _, v in cv.split(X_train, y_train)],
                 "pr_auc_mean": scores.mean(), "pr_auc_std": scores.std()})

skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE)   # the SAME folds from now on (and in 02)
pd.DataFrame(rows)

<a name="11"></a>
## 11. ROC, PR y threshold (hands-on 2)

El modelo entrega un **score**; la **decisión** aparece con un threshold $\tau$. Para elegirlo sin tocar el test se usan probabilidades **out-of-fold**: cada caso de train es puntuado por un modelo que no lo vio.

In [ ]:
# =============================================================================
# 11.1 ROC and PR curves, threshold sweep and expected cost (out-of-fold)
# =============================================================================
oof = cross_val_predict(build_pipeline(), X_train, y_train, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
thresholds = np.round(np.concatenate([np.arange(0.01, 0.1, 0.01), np.arange(0.1, 1.0, 0.05)]), 3)
sweep = pd.DataFrame([binary_report(y_train, oof, t) for t in thresholds])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fpr, tpr, _ = roc_curve(y_train, oof)
axes[0].plot(fpr, tpr)
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title=f"ROC (AUC={roc_auc_score(y_train, oof):.3f})", xlabel="FPR", ylabel="recall")
prec, rec, _ = precision_recall_curve(y_train, oof)
axes[1].plot(rec, prec)
axes[1].axhline(y_train.mean(), ls="--", color="gray")
axes[1].set(title=f"PR (AP={average_precision_score(y_train, oof):.3f})", xlabel="recall", ylabel="precision")
axes[2].plot(sweep["threshold"], sweep["expected_cost"], color="C3")
axes[2].axvline(C_FP / (C_FP + C_FN), ls=":", color="gray", label="τ* teórico")
axes[2].set(title="Costo esperado según τ", xlabel="threshold τ")
axes[2].legend()
plt.tight_layout()
plt.show()
sweep[["threshold", "precision", "recall", "alerts", "expected_cost"]].iloc[::3]

In [ ]:
# =============================================================================
# 11.2 TunedThresholdClassifierCV: the minimum-cost threshold with internal CV
# =============================================================================
cost_scorer = make_scorer(lambda y_true, y_pred: -expected_cost(y_true, y_pred))
tuned = TunedThresholdClassifierCV(build_pipeline(), scoring=cost_scorer, cv=skf, n_jobs=-1).fit(X_train, y_train)
print(f"tuned threshold: {tuned.best_threshold_:.3f} | sweep minimum: {sweep.loc[sweep['expected_cost'].idxmin(), 'threshold']:.3f}"
      f" | theory tau* = C_FP/(C_FP+C_FN) = {C_FP / (C_FP + C_FN):.3f}")

**Lectura técnica**

- La ROC se ve excelente porque con 99 % de negativos muchos falsos positivos son poca FPR. **La curva PR es la que muestra el costo de ganar recall.**
- El $\tau^* \approx 0{,}02$ teórico solo vale si las probabilidades están **calibradas**. La diferencia con el threshold elegido por CV es una pista de que no lo están (ver calibración en las slides y el ejercicio 2).
- El threshold también fija **cuántas alertas** revisa el equipo.

<a name="12"></a>
## 12. Búsqueda de hiperparámetros (hands-on 3)

El **preprocesamiento también tiene hiperparámetros**: el percentil del clip se elige con CV igual que `C` o `class_weight`. Grid frente a random con el **mismo presupuesto**:

In [ ]:
# =============================================================================
# 12.1 Grid vs random search with the same budget
# =============================================================================
search_cv = StratifiedKFold(3 if FAST_DEMO_MODE else N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(build_pipeline(), {                                  # 3 x 2 x 2 = 12 candidates
    "model__C": [0.01, 0.1, 1.0], "model__class_weight": [None, "balanced"],
    "prep__columns__num__clip__upper": [0.99, 0.999]}, scoring="average_precision", cv=search_cv, n_jobs=-1)
random = RandomizedSearchCV(build_pipeline(), {
    "model__C": loguniform(1e-3, 1e2), "model__class_weight": [None, "balanced"],
    "prep__columns__num__clip__upper": [0.95, 0.99, 0.995, 0.999, None]},
    n_iter=12, scoring="average_precision", cv=search_cv, n_jobs=-1, random_state=RANDOM_STATE)

searches = {"grid": grid.fit(X_train, y_train), "random": random.fit(X_train, y_train)}
pd.DataFrame([{"search": k, "best_pr_auc": s.best_score_, "std_between_folds": s.cv_results_["std_test_score"][s.best_index_],
               "best_params": s.best_params_} for k, s in searches.items()])

**Lectura técnica:** compare con la PR-AUC de la sección 10. El salto viene sobre todo del **clip al percentil 99**: con montos legítimos de miles, un modelo lineal sin clip aprende mal la relación entre monto y fraude. El mejor score de una búsqueda es **optimista** (es el máximo de muchos intentos): el test mide el procedimiento completo.

<a name="13"></a>
## 13. Modelo final, evaluación única en test y artefacto

Con hiperparámetros y threshold decididos **solo con train**, se abre el test **una vez**. Luego se guarda **el pipeline completo** (nunca solo el modelo: sin el preprocesamiento, los datos nuevos llegarían con otra escala y otras columnas) junto a una **model card**.

In [ ]:
# =============================================================================
# 13.1 Refit with the chosen configuration and threshold, evaluate once
# =============================================================================
final_params = max(searches.values(), key=lambda s: s.best_score_).best_params_
final_tuned = TunedThresholdClassifierCV(build_pipeline().set_params(**final_params), scoring=cost_scorer,
                                         cv=skf, n_jobs=-1).fit(X_train, y_train)
final_model, final_threshold = final_tuned.estimator_, float(final_tuned.best_threshold_)

proba_test = final_model.predict_proba(X_test)[:, 1]
test_report = binary_report(y_test, proba_test, final_threshold, "final (tuned τ)")
ConfusionMatrixDisplay(confusion_matrix(y_test, proba_test >= final_threshold), display_labels=["legit", "fraud"]).plot(colorbar=False)
plt.title(f"Modelo final en test (τ = {final_threshold:.3f})")
plt.grid(False)
plt.show()
comparison = pd.concat([baselines, pd.DataFrame([test_report])], ignore_index=True)
comparison

In [ ]:
# =============================================================================
# 13.2 Save pipeline + model card (+ baseline metrics for notebook 02)
# =============================================================================
MODEL_PATH, CARD_PATH, BASELINE_PATH = (ARTIFACT_DIR / "fraud_pipeline_v1.joblib", ARTIFACT_DIR / "model_card.json",
                                        ARTIFACT_DIR / "baseline_metrics.csv")
joblib.dump(final_model, MODEL_PATH)
model_card = {
    "model_name": "fraud_pipeline_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "input_schema": INPUT_SCHEMA,
    "features_code": "notebook 01, cell 7.1 (SHARED PREP BLOCK 2/3): must be defined before joblib.load",
    "model_features": list(final_model.named_steps["prep"].get_feature_names_out()),
    "threshold": final_threshold,
    "costs": {"C_FN": C_FN, "C_FP": C_FP},
    "params": {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in final_params.items()},
    "training": {"rows": int(len(X_train)), "fraud_rate": float(y_train.mean()), "fast_demo_mode": FAST_DEMO_MODE},
    "test_metrics": {k: float(v) for k, v in test_report.items() if isinstance(v, (int, float, np.floating))},
    "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
CARD_PATH.write_text(json.dumps(model_card, indent=2, ensure_ascii=False))
comparison.to_csv(BASELINE_PATH, index=False)
print(f"saved to {ARTIFACT_DIR.resolve()}: {MODEL_PATH.name}, {CARD_PATH.name}, {BASELINE_PATH.name}")

if IN_COLAB and not USE_GOOGLE_DRIVE:          # /content is wiped: download to keep them
    from google.colab import files
    for path in (MODEL_PATH, CARD_PATH, BASELINE_PATH):
        files.download(str(path))

<a name="14"></a>
## 14. Inferencia sobre datos desconocidos

Simulamos el servicio desplegado: estas celdas **solo** usan el `.joblib`, la model card, las clases de la celda 7.1 y datos crudos nuevos. Nada del entrenamiento que esté en memoria.

In [ ]:
# =============================================================================
# 14.1 The "production service": load the artifact and score raw rows
# =============================================================================
service_model = joblib.load(ARTIFACT_DIR / "fraud_pipeline_v1.joblib")
service_card = json.loads((ARTIFACT_DIR / "model_card.json").read_text())


def predict_fraud(raw: pd.DataFrame) -> pd.DataFrame:
    proba = service_model.predict_proba(validate_schema(raw))[:, 1]
    return pd.DataFrame({"fraud_probability": proba, "alert": proba >= service_card["threshold"]}, index=raw.index)


new_transaction = {                         # e.g. the JSON body of an API request
    "trans_date_trans_time": "2020-12-24 23:41:10", "category": "shopping_net", "amt": 912.35, "gender": "M",
    "state": "TX", "lat": 29.76, "long": -95.37, "city_pop": 2_300_000, "dob": "1961-04-02",
    "merch_lat": 30.10, "merch_long": -95.80, "customer_note": "not used by the model",
}
print("reloaded artifact == in-memory model:", np.allclose(predict_fraud(X_test)["fraud_probability"], proba_test))
predict_fraud(pd.DataFrame([new_transaction]))

In [ ]:
# =============================================================================
# 14.2 Same scale and dimensions with unseen values, nulls, extremes — and a missing column
# =============================================================================
edge_cases = pd.DataFrame([
    {**new_transaction, "category": "crypto_exchange"},   # never seen in training
    {**new_transaction, "state": None},                   # missing value
    {**new_transaction, "amt": 1_000_000.0},               # extreme: clipped with the train limit
])
features = service_model.named_steps["prep"].transform(validate_schema(edge_cases))
print(f"shape {features.shape} | same columns and order as training: {list(features.columns) == service_card['model_features']}")
display(predict_fraud(edge_cases).join(features[["amt"]].rename(columns={"amt": "amt_transformed"})))

try:
    predict_fraud(pd.DataFrame([new_transaction]).drop(columns=["amt"]))
except ValueError as err:
    print("rejected:", err)

**Lectura técnica**

- **Categoría nueva:** el one-hot la codifica como ceros (o como "infrecuente"): no aparece una columna nueva.
- **Nulo:** se imputa con el valor aprendido en train.
- **Monto extremo:** el clip lo limita al percentil de train. Como 912 ya lo supera, las tres filas quedan con el mismo `amt` transformado.
- **Columna faltante:** se rechaza con un mensaje claro, mejor que una predicción silenciosamente mala.

Pase lo que pase, el modelo recibe **las mismas columnas, en el mismo orden y con la escala de entrenamiento**.

In [ ]:
# =============================================================================
# 14.3 Six months of future transactions (fraudTest.csv)
# =============================================================================
production = load_fraud_csv(DATA_DIR / FILES["production"])
t0 = time.perf_counter()
scored = predict_fraud(production.drop(columns=[TARGET]))
print(f"scored {len(production):,} transactions in {time.perf_counter() - t0:.1f}s")
print(f"fraud rate: train {y_train.mean():.3%} -> production {production[TARGET].mean():.3%}")
future_report = binary_report(production[TARGET], scored["fraud_probability"], service_card["threshold"], "production (future)")
pd.DataFrame([test_report, future_report]).set_index("model")[["precision", "recall", "pr_auc", "roc_auc", "alerts"]]

**Lectura técnica:** en datos futuros la PR-AUC baja respecto al test interno. El test interno era del **mismo periodo y de los mismos clientes** que train, así que era optimista; además, la **prevalencia** cambia entre periodos, y con ella la precision y el volumen de alertas del mismo threshold. Por eso un modelo desplegado se **monitorea**.

<a name="15"></a>
## 15. Ejercicios y puente a la parte 2

**Ejercicios**

1. **Costo por monto:** cambie $C_{FN}=500$ por el monto de la transacción. ¿Cambia el threshold óptimo?
2. **Calibración:** compare la curva de confiabilidad del modelo con `CalibratedClassifierCV` (sigmoid e isotonic). ¿Se acerca el threshold elegido a $\tau^*$?
3. **Capacidad:** si el equipo revisa 200 alertas por día, ¿qué threshold usar en `fraudTest` y qué recall se obtiene?
4. **Validación temporal:** ordene la historia por fecha y use `TimeSeriesSplit`. ¿El score se acerca al de `fraudTest`?

**La frase que debe sobrevivir:** un modelo merece confianza cuando el baseline es serio, la métrica es correcta, la validación no tiene leakage, el threshold está razonado y **el pipeline completo funciona igual sobre datos que nunca vio**.

**Parte 2 (`02_ml_boosting_fraude.ipynb`):** copia las celdas `SHARED PREP BLOCK` 1/3 (datos y split), 2/3 (features) y 3/3 (pipeline), usa los mismos folds `skf` y solo cambia el `model`: árboles, ensembles, boosting, SMOTE frente a `class_weight` y Optuna. Si encuentra `baseline_metrics.csv` y `model_card.json` los usa como barra a superar; si no (p. ej. Colab sin Drive), reconstruye el baseline en menos de un minuto.